In [ ]:
#Parte 1
#Importar Librerias
import pandas as pd
import sqlite3
import json
from pathlib import Path

print("Librerias importadas")

Librerias importadas


In [ ]:
#Definicion de rutas de trabajo
BASE_DIR = Path("/content")
CSV_PATH = BASE_DIR /"datos/produccion_energetica.csv"
DB_PATH  = BASE_DIR /"datos/sensores.db"
JSON_PATH = BASE_DIR /"datos/mantenimiento.json"

print(f"Rutas creadas")

Rutas creadas


In [ ]:
#Cargar CSV
df_produccion = pd.read_csv(CSV_PATH)
print("--- PRODUCCIÓN (CSV) ---")
print("Forma: ", df_produccion.shape)
print(df_produccion.dtypes)
print(df_produccion.head())

--- PRODUCCIÓN (CSV) ---
Forma:  (50966, 7)
id                      int64
fecha                  object
panel_id               object
region                 object
produccion_kwh        float64
temperatura_c         float64
irradiancia_kwh_m2    float64
dtype: object
      id       fecha panel_id      region  produccion_kwh  temperatura_c  \
0  26714  2024-08-05  PNL-001    USULUTAN           26.34           17.4   
1  40117  2025-05-21  PNL-010   Sonsonate           64.30           35.0   
2  18177  2024-01-31  PNL-020  SAN MIGUEL             NaN           28.8   
3  17859  2024-01-24  PNL-001    Usulutan           22.47           23.1   
4  49621  2025-12-13  PNL-049  Santa Ana            18.84           30.6   

   irradiancia_kwh_m2  
0                6.28  
1                6.18  
2                5.66  
3                6.86  
4                4.57  


In [ ]:
# Conectar a la base de datos SQLite y consultar la tabla sensores
conexion = sqlite3.connect(DB_PATH)
df_sensores = pd.read_sql_query("SELECT * FROM sensores;", conexion)
conexion.close()

print("\n--- SENSORES EN LA BASE DE DATOS ---")
print("Forma:", df_sensores.shape)
print(df_sensores.dtypes)
print(df_sensores.head())


--- SENSORES EN LA BASE DE DATOS ---
Forma: (77, 9)
sensor_id             object
panel_id              object
tipo_sensor           object
region                object
latitud              float64
longitud             float64
capacidad_kw         float64
fecha_instalacion     object
numero_serie          object
dtype: object
   sensor_id panel_id  tipo_sensor       region    latitud   longitud  \
0  SEN-001-1  PNL-001  Irradiancia     Usulután  13.244730 -88.320279   
1  SEN-001-2  PNL-001   Producción     Usulután  13.244730 -88.320279   
2  SEN-002-1  PNL-002  Irradiancia     Usulután  13.233244 -88.322397   
3  SEN-003-1  PNL-003  Irradiancia  La Libertad  13.141317 -89.875131   
4  SEN-003-2  PNL-003  Irradiancia  La Libertad  13.141317 -89.875131   

   capacidad_kw fecha_instalacion numero_serie  
0           5.2        2022-07-13     SN311037  
1           5.2        2022-07-13     SN474738  
2           7.9        2021-04-07     SN202805  
3           5.1        2021-03-26    

In [ ]:
# Leer archivo JSON
df_mantenimiento = pd.read_json(JSON_PATH)

print("\n--- HISTORIAL DE MANTENIMIENTO (JSON) ---")
print("Forma:", df_mantenimiento.shape)
print(df_mantenimiento.dtypes)
print(df_mantenimiento.head())


--- HISTORIAL DE MANTENIMIENTO (JSON) ---
Forma: (263, 8)
mantenimiento_id        object
equipo_id               object
fecha                   object
tipo_mantenimiento      object
tecnico_responsable     object
duracion_horas         float64
costo_usd              float64
observaciones           object
dtype: object
  mantenimiento_id equipo_id       fecha       tipo_mantenimiento  \
0        MNT-00001   PNL-022  2023-01-05     Inspección de rutina   
1        MNT-00002   PNL-038  2023-01-05  Reemplazo de componente   
2        MNT-00003   PNL-001  2023-01-11               Preventivo   
3        MNT-00004   PNL-002  2023-01-18        Limpieza de panel   
4        MNT-00005   PNL-003  2023-01-18               Correctivo   

         tecnico_responsable  duracion_horas  costo_usd  \
0    Fátima Beatriz Alvarado             3.4     200.21   
1  Gabriela Alexandra Molina             2.4      32.10   
2         María Elena Guzmán             4.8     357.71   
3         María Elena Guzmán

In [ ]:
#Integración de los 3 archivos
df_mantenimiento = df_mantenimiento.rename(columns={"equipo_id": "panel_id"})
df_sensores_unicos = df_sensores.drop_duplicates(subset=["panel_id"])
df_integrado = pd.merge(
    df_produccion, df_sensores_unicos, on="panel_id", how="left"
)
df_final = pd.merge(
    df_integrado, df_mantenimiento, on=["panel_id", "fecha"], how="left"
)

print("\n--- DATASET FINAL CONSOLIDADO ---")
print(f"Forma final: {df_final.shape} (Filas, Columnas)")
df_final.head()


--- DATASET FINAL CONSOLIDADO ---
Forma final: (50969, 21) (Filas, Columnas)


,id,fecha,panel_id,region_x,produccion_kwh,temperatura_c,irradiancia_kwh_m2,sensor_id,tipo_sensor,region_y,...,longitud,capacidad_kw,fecha_instalacion,numero_serie,mantenimiento_id,tipo_mantenimiento,tecnico_responsable,duracion_horas,costo_usd,observaciones
0,26714,2024-08-05,PNL-001,USULUTAN,26.34,17.4,6.28,SEN-001-1,Irradiancia,Usulután,...,-88.320279,5.2,2022-07-13,SN311037,NaN,NaN,NaN,NaN,NaN,NaN
1,40117,2025-05-21,PNL-010,Sonsonate,64.30,35.0,6.18,SEN-010-1,Irradiancia,Sonsonate,...,-89.933068,9.0,2020-10-02,SN776779,NaN,NaN,NaN,NaN,NaN,NaN
2,18177,2024-01-31,PNL-020,SAN MIGUEL,NaN,28.8,5.66,SEN-020-1,Irradiancia,San Miguel,...,-89.071759,8.3,2021-11-27,SN354091,NaN,NaN,NaN,NaN,NaN,NaN
3,17859,2024-01-24,PNL-001,Usulutan,22.47,23.1,6.86,SEN-001-1,Irradiancia,Usulután,...,-88.320279,5.2,2022-07-13,SN311037,NaN,NaN,NaN,NaN,NaN,NaN
4,49621,2025-12-13,PNL-049,Santa Ana,18.84,30.6,4.57,SEN-049-1,Consumo,Santa Ana,...,-88.934461,4.9,2021-10-18,SN590241,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Definir ruta de guardado para el dataset limpio
ruta_dataset_limpio = BASE_DIR / "dataset_limpio_Cristina_Fuentes.csv"

# Guardar en formato CSV sin el índice
df_final.to_csv(ruta_dataset_limpio, index=False, encoding="utf-8")

print(f"Dataset limpio guardado exitosamente en: {ruta_dataset_limpio}")

Dataset limpio guardado exitosamente en: /content/dataset_limpio_Cristina_Fuentes.csv


In [ ]:
#Descargar dataset limpio

from google.colab import files
files.download(BASE_DIR / "dataset_limpio_Cristina_Fuentes.csv")

print("Archivo descargado")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Archivo descargado


In [ ]:
# ¿Qué llave(s) usaste para unir las 3 fuentes y qué problema
# concreto encontraste al consolidarlas (ej. tipos de dato distintos, IDs no coincidentes)? Menciona la forma final
# (filas × columnas) de tu DataFrame consolidado.

# Para poder unir las tres fuentes se utilizó la llave panel_id (uniendo con df_sensores) y la combinación de llaves
# ['panel_id', 'fecha'] para integrar los eventos de mantenimiento diarios. Al unirlos, el principal problema
# detectado fue la diferencia en el nombre de la llave del archivo JSON (equipo_id en lugar de panel_id),
# obligando su renombrado, además de la existencia de múltiples sensores por panel en SQLite que exigieron
# duplicación para evitar registros innecesarios. El DataFrame final consolidado quedó con una forma exacta de 50,969 filas y 21 columnas.

In [ ]:
# Parte 2
# Crear copias de trabajo
df_original = df_final.copy()
df_limpio = df_final.copy()

print("Copias de trabajo creadas: -------")
print(f"Dimensiones iniciales: {df_limpio.shape[0]} filas, {df_limpio.shape[1]} columnas\n")

# Identificar valores nulos y duplicados
print(f"Registros duplicados encontrados: {df_limpio.duplicated().sum()}")
print("\nValores nulos en mediciones clave:")
print(df_limpio[["produccion_kwh", "temperatura_c", "irradiancia_kwh_m2"]].isnull().sum())

Copias de trabajo creadas: -------
Dimensiones iniciales: 50969 filas, 21 columnas

Registros duplicados encontrados: 505

Valores nulos en mediciones clave:
produccion_kwh        1536
temperatura_c         1017
irradiancia_kwh_m2     762
dtype: int64


In [ ]:
#Diagnostico inicial

#Cuantos datos hay
print("Dimensiones: ------")
print(f"Filas: {df_limpio.shape[0]}, Columnas: {df_limpio.shape[1]}\n")

#Nombres de columnas
print("Nombres de las columnas: ------")
print(df_limpio.columns.tolist())

Dimensiones: ------
Filas: 50969, Columnas: 21

Nombres de las columnas: ------
['id', 'fecha', 'panel_id', 'region_x', 'produccion_kwh', 'temperatura_c', 'irradiancia_kwh_m2', 'sensor_id', 'tipo_sensor', 'region_y', 'latitud', 'longitud', 'capacidad_kw', 'fecha_instalacion', 'numero_serie', 'mantenimiento_id', 'tipo_mantenimiento', 'tecnico_responsable', 'duracion_horas', 'costo_usd', 'observaciones']


In [ ]:
# Registros duplicaods

print("Registros duplicados: ---------")

duplicados = df_limpio.duplicated().sum()
print(f"Registros duplicadas: {duplicados}")


Registros duplicados: ---------
Registros duplicadas: 505


In [ ]:
# Eliminar registros duplicados
print(f"Duplicados antes: {df_limpio.duplicated().sum()}")
df_limpio = df_limpio.drop_duplicates().reset_index(drop=True)
print(f"Duplicados después: {df_limpio.duplicated().sum()}")
print(f"Dimensiones tras deduplicar: {df_limpio.shape}")

Duplicados antes: 505
Duplicados después: 0
Dimensiones tras deduplicar: (50464, 21)


In [ ]:
# Correguir valores negativos en produccion
negativos = (df_limpio["produccion_kwh"] < 0).sum()
print(f"Producciones negativas detectadas: {negativos}")
if negativos > 0:
  df_limpio["produccion_kwh"] = df_limpio["produccion_kwh"].abs()
  print("Valores negativos corregidos con valor absoluto (.abs())")

Producciones negativas detectadas: 149
Valores negativos corregidos con valor absoluto (.abs())


In [ ]:
# Atribucion por mediana en variables continuas
med_prod = df_limpio["produccion_kwh"].median()
med_temp = df_limpio["temperatura_c"].median()
med_irr = df_limpio["irradiancia_kwh_m2"].median()

print(f"\nMedianas calculadas -> Producción: {med_prod:.2f}, Temp: {med_temp:.2f}, Irradiancia: {med_irr:.2f}")

df_limpio["produccion_kwh"] = df_limpio["produccion_kwh"].fillna(med_prod)
df_limpio["temperatura_c"] = df_limpio["temperatura_c"].fillna(med_temp)
df_limpio["irradiancia_kwh_m2"] = df_limpio["irradiancia_kwh_m2"].fillna(med_irr)


Medianas calculadas -> Producción: 28.54, Temp: 27.00, Irradiancia: 5.41


In [ ]:
# Atribucion en columnas de mantenimiento (dias sin evento)
df_limpio["mantenimiento_id"] = df_limpio["mantenimiento_id"].fillna(
    "Sin registro"
)
df_limpio["tipo_mantenimiento"] = df_limpio["tipo_mantenimiento"].fillna(
    "Sin mantenimiento"
)
df_limpio["tecnico_responsable"] = df_limpio["tecnico_responsable"].fillna(
    "Ninguno"
)
df_limpio["duracion_horas"] = df_limpio["duracion_horas"].fillna(0)
df_limpio["costo_usd"] = df_limpio["costo_usd"].fillna(0)
df_limpio["observaciones"] = df_limpio["observaciones"].fillna(
    "Sin observaciones"
)

print("\nNulos restantes en produccion_kwh:", df_limpio["produccion_kwh"].isnull().sum())


Nulos restantes en produccion_kwh: 0


In [ ]:
# Normalizar columna de Región
mapeo_regiones = {
    'usulutan': 'Usulután', 'usulután': 'Usulután',
    'santa ana': 'Santa Ana', 'sta. ana': 'Santa Ana',
    'san salvador': 'San Salvador',
    'la libertad': 'La Libertad',
    'san miguel': 'San Miguel',
    'sonsonate': 'Sonsonate'
}

# Limpiar espacios en blanco antes de mapear
if 'region_x' in df_limpio.columns:
    df_limpio['region'] = df_limpio['region_x'].astype(str).str.strip().str.lower().map(mapeo_regiones)
    df_limpio = df_limpio.drop(columns=['region_x', 'region_y'], errors='ignore')
else:
    # Por si se vuelve a correr sobre df_limpio ya procesado
    df_limpio['region'] = df_final['region_x'].astype(str).str.strip().str.lower().map(mapeo_regiones)

print("Regiones únicas normalizadas (Total:", df_limpio["region"].notnull().sum(), "):")
print(df_limpio["region"].value_counts(dropna=False))

Regiones únicas normalizadas (Total: 50464 ):
region
Usulután        10122
Sonsonate        9081
San Salvador     8089
La Libertad      8063
Santa Ana        8057
San Miguel       7052
Name: count, dtype: int64


In [ ]:
# Convertir fecha a datetime
df_limpio["fecha"] = pd.to_datetime(df_limpio["fecha"], errors="coerce")
print(df_limpio.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50464 entries, 0 to 50463
Data columns (total 20 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   id                   50464 non-null  int64         
 1   fecha                50464 non-null  datetime64[ns]
 2   panel_id             50464 non-null  object        
 3   produccion_kwh       50464 non-null  float64       
 4   temperatura_c        50464 non-null  float64       
 5   irradiancia_kwh_m2   50464 non-null  float64       
 6   sensor_id            50464 non-null  object        
 7   tipo_sensor          50464 non-null  object        
 8   latitud              50464 non-null  float64       
 9   longitud             50464 non-null  float64       
 10  capacidad_kw         50464 non-null  float64       
 11  fecha_instalacion    50464 non-null  object        
 12  numero_serie         50464 non-null  object        
 13  mantenimiento_id     50464 non-

In [ ]:
# convertir columnas numericas
columnas_num = [
    "produccion_kwh",
    "temperatura_c",
    "irradiancia_kwh_m2",
    "capacidad_kw",
    "latitud",
    "longitud",
    "duracion_horas",
    "costo_usd",
]
for col in columnas_num:
  if col in df_limpio.columns:
    df_limpio[col] = pd.to_numeric(df_limpio[col], errors="coerce")

# Caracteristicas temporales
df_limpio["anio"] = df_limpio["fecha"].dt.year
df_limpio["mes"] = df_limpio["fecha"].dt.month
df_limpio["dia"] = df_limpio["fecha"].dt.day
df_limpio["dia_semana"] = df_limpio["fecha"].dt.day_name()
df_limpio["es_fin_semana"] = df_limpio["fecha"].dt.day_of_week >= 5
print(df_limpio.info())



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50464 entries, 0 to 50463
Data columns (total 25 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   id                   50464 non-null  int64         
 1   fecha                50464 non-null  datetime64[ns]
 2   panel_id             50464 non-null  object        
 3   produccion_kwh       50464 non-null  float64       
 4   temperatura_c        50464 non-null  float64       
 5   irradiancia_kwh_m2   50464 non-null  float64       
 6   sensor_id            50464 non-null  object        
 7   tipo_sensor          50464 non-null  object        
 8   latitud              50464 non-null  float64       
 9   longitud             50464 non-null  float64       
 10  capacidad_kw         50464 non-null  float64       
 11  fecha_instalacion    50464 non-null  object        
 12  numero_serie         50464 non-null  object        
 13  mantenimiento_id     50464 non-

In [ ]:
# Variables derivadas especificas
# Rendimiento específico (energía producida por cada kW instalado)
df_limpio["rendimiento_especifico"] = (
    df_limpio["produccion_kwh"] / df_limpio["capacidad_kw"]
).round(4)
print(df_limpio.head())

      id      fecha panel_id  produccion_kwh  temperatura_c  \
0  26714 2024-08-05  PNL-001           26.34           17.4   
1  40117 2025-05-21  PNL-010           64.30           35.0   
2  18177 2024-01-31  PNL-020           28.54           28.8   
3  17859 2024-01-24  PNL-001           22.47           23.1   
4  49621 2025-12-13  PNL-049           18.84           30.6   

   irradiancia_kwh_m2  sensor_id  tipo_sensor    latitud   longitud  ...  \
0                6.28  SEN-001-1  Irradiancia  13.244730 -88.320279  ...   
1                6.18  SEN-010-1  Irradiancia  13.349975 -89.933068  ...   
2                5.66  SEN-020-1  Irradiancia  14.071118 -89.071759  ...   
3                6.86  SEN-001-1  Irradiancia  13.244730 -88.320279  ...   
4                4.57  SEN-049-1      Consumo  13.316204 -88.934461  ...   

   duracion_horas costo_usd      observaciones      region  anio mes  dia  \
0             0.0       0.0  Sin observaciones    Usulután  2024   8    5   
1         

In [ ]:
# Indicador binario si hubo mantenimiento
df_limpio["hubo_mantenimiento"] = (
    df_limpio["tipo_mantenimiento"] != "Sin mantenimiento"
).astype(int)

print("Variables derivadas creadas:")
display(
    df_limpio[
        [
            "fecha",
            "panel_id",
            "produccion_kwh",
            "rendimiento_especifico",
            "hubo_mantenimiento",
            "anio",
            "mes",
        ]
    ].head()
)

Variables derivadas creadas:


,fecha,panel_id,produccion_kwh,rendimiento_especifico,hubo_mantenimiento,anio,mes
0,2024-08-05,PNL-001,26.34,5.0654,0,2024,8
1,2025-05-21,PNL-010,64.30,7.1444,0,2025,5
2,2024-01-31,PNL-020,28.54,3.4386,0,2024,1
3,2024-01-24,PNL-001,22.47,4.3212,0,2024,1
4,2025-12-13,PNL-049,18.84,3.8449,0,2025,12


In [ ]:
# Definir ruta de guardado para el dataset limpio
ruta_archivo_limpio = (
    BASE_DIR / "dataset_limpio_parte2_Cristina_Fuentes.csv"
)

# Guardar df_limpio
df_limpio.to_csv(ruta_archivo_limpio, index=False, encoding="utf-8")
print(f"Archivo guardado exitosamente en: {ruta_archivo_limpio}")

Archivo guardado exitosamente en: /content/dataset_limpio_parte2_Cristina_Fuentes.csv


In [ ]:
#Descargar dataset limpio

from google.colab import files
files.download(BASE_DIR / "dataset_limpio_parte2_Cristina_Fuentes.csv")

print("Archivo descargado")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Archivo descargado


In [ ]:
# Indica la cantidad EXACTA de nulos y duplicados que encontraste en tu ejecución,
# qué método de imputación usaste (ej. media, mediana, eliminación) y por qué lo preferiste sobre otra alternativa

# En la ejecución se identificaron y eliminaron exactamente 505 registros duplicados. En las variables de medición se detectaron 1,514 valores nulos
# en produccion_kwh, 1,009 en temperatura_c y 757 en irradiancia_kwh_m2. Se utilizó la imputación por mediana (28.54 kWh, 27.0 °C y 5.41 kWh/m² respectivamente)
# en lugar de la media o eliminación directa, ya que las series climáticas y de generación presentan asimetría y lecturas atípicas, evitando así distorsiones
# o pérdida innecesaria de registros históricos. Se normalizó la columna region eliminando inconsistencias ortográficas y espacios, y se crearon variables
# derivadas temporales (anio, mes, dia_semana, es_fin_semana) y operativas (rendimiento_especifico y hubo_mantenimiento)

In [ ]:
# Parte 3
# Instalar spark
%pip install pyspark
print("------- PySpark instalado -------")

------- PySpark instalado -------


In [ ]:
# Importar libreria de Spark
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

print("------ Librerías importadas ------")

------ Librerías importadas ------


In [ ]:
# Crear sesión de spark
spark = SparkSession.builder \
.master("local[*]")\
.appName("GreenEnergy_Cristina_Fuentes")\
.getOrCreate()

print(f"Spark version: {spark.version}")
print(f"Paralelismo {spark.sparkContext.defaultParallelism}")

Spark version: 4.0.4
Paralelismo 2


In [ ]:
# Definir rutas
BASE_DIR = Path("/content")
DATA_PROCESSED = BASE_DIR/"datos"/"procesados"
DATA_ANALYTICS = BASE_DIR/"datos"/"analiticos"

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
DATA_ANALYTICS.mkdir(parents=True, exist_ok=True)

print(f"Datos procesados:m {DATA_PROCESSED}")
print(f"Datos analíticos: {DATA_ANALYTICS}")

Datos procesados:m /content/datos/procesados
Datos analíticos: /content/datos/analiticos


In [ ]:
# Cargar dataset limpio en Spark
ruta_entrada = BASE_DIR / "dataset_limpio_parte2_Cristina_Fuentes.csv"

df_spark = (
    spark.read.option("header", True)
    .option("inferSchema", True)
    .csv(str(ruta_entrada))
)

print("------ Dataset cargado en Spark ------")
print(f"Filas: {df_spark.count()}")
print(f"Columnas: {len(df_spark.columns)}")
print("Columnas detectadas por Spark:\n", df_spark.columns)

------ Dataset cargado en Spark ------
Filas: 50464
Columnas: 27
Columnas detectadas por Spark:
 ['id', 'fecha', 'panel_id', 'produccion_kwh', 'temperatura_c', 'irradiancia_kwh_m2', 'sensor_id', 'tipo_sensor', 'latitud', 'longitud', 'capacidad_kw', 'fecha_instalacion', 'numero_serie', 'mantenimiento_id', 'tipo_mantenimiento', 'tecnico_responsable', 'duracion_horas', 'costo_usd', 'observaciones', 'region', 'anio', 'mes', 'dia', 'dia_semana', 'es_fin_semana', 'rendimiento_especifico', 'hubo_mantenimiento']


In [ ]:
# Verificar esquema y primeras filas
df_spark.printSchema()
df_spark.show(5, truncate=False)

root
 |-- id: integer (nullable = true)
 |-- fecha: date (nullable = true)
 |-- panel_id: string (nullable = true)
 |-- produccion_kwh: double (nullable = true)
 |-- temperatura_c: double (nullable = true)
 |-- irradiancia_kwh_m2: double (nullable = true)
 |-- sensor_id: string (nullable = true)
 |-- tipo_sensor: string (nullable = true)
 |-- latitud: double (nullable = true)
 |-- longitud: double (nullable = true)
 |-- capacidad_kw: double (nullable = true)
 |-- fecha_instalacion: date (nullable = true)
 |-- numero_serie: string (nullable = true)
 |-- mantenimiento_id: string (nullable = true)
 |-- tipo_mantenimiento: string (nullable = true)
 |-- tecnico_responsable: string (nullable = true)
 |-- duracion_horas: double (nullable = true)
 |-- costo_usd: double (nullable = true)
 |-- observaciones: string (nullable = true)
 |-- region: string (nullable = true)
 |-- anio: integer (nullable = true)
 |-- mes: integer (nullable = true)
 |-- dia: integer (nullable = true)
 |-- dia_semana: s

In [ ]:
# Transformacion 1: filtro distribuido
df_filtrado = df_spark.filter(
    (F.col("produccion_kwh") > 0) & (F.col("irradiancia_kwh_m2") > 0)
)
print(f"Registros tras el filtro: {df_filtrado.count()}")

Registros tras el filtro: 50464


In [ ]:
# Transformacion 2: agregacion distribuida
resumen_region = (
    df_filtrado.groupBy("region")
    .agg(
        F.round(F.sum("produccion_kwh"), 2).alias("produccion_total_kwh"),
        F.round(F.avg("produccion_kwh"), 2).alias("produccion_promedio_kwh"),
        F.round(F.avg("irradiancia_kwh_m2"), 2).alias(
            "irradiancia_promedio_kwh_m2"
        ),
        F.count("*").alias("total_mediciones"),
    )
    .orderBy(F.desc("produccion_total_kwh"))
)

print("\n------ Agregación Spark: Producción energética por Región ------")
resumen_region.show(truncate=False)


------ Agregación Spark: Producción energética por Región ------
+------------+--------------------+-----------------------+---------------------------+----------------+
|region      |produccion_total_kwh|produccion_promedio_kwh|irradiancia_promedio_kwh_m2|total_mediciones|
+------------+--------------------+-----------------------+---------------------------+----------------+
|Usulután    |296969.5            |29.34                  |5.41                       |10122           |
|Sonsonate   |296183.79           |32.62                  |5.41                       |9081            |
|La Libertad |272584.33           |33.81                  |5.43                       |8063            |
|Santa Ana   |231752.81           |28.76                  |5.39                       |8057            |
|San Miguel  |224760.21           |31.87                  |5.4                        |7052            |
|San Salvador|188417.33           |23.29                  |5.41                       |8089   

In [ ]:
# Crear vista temporal
df_spark.createOrReplaceTempView("energia_analytics")
print("Vista temporal 'energia_analytics' creada exitosamente.")

# Consulta de spark sql del negocio
# Pregunta de negocio: ¿Cuál es el desempeño energético total, promedio y el rendimiento técnico por departamento?
consulta_sql = """
    SELECT
        region,
        COUNT(*) AS mediciones_totales,
        ROUND(SUM(produccion_kwh), 2) AS produccion_total_kwh,
        ROUND(AVG(produccion_kwh), 2) AS promedio_generacion_diaria,
        ROUND(AVG(rendimiento_especifico), 2) AS rendimiento_tecnico_promedio
    FROM energia_analytics
    GROUP BY region
    ORDER BY produccion_total_kwh DESC
"""

resultado_sql = spark.sql(consulta_sql)
print("\n------ RESULTADO SPARK SQL: DESEMPEÑO POR REGIÓN ------")
resultado_sql.show(truncate=False)

Vista temporal 'energia_analytics' creada exitosamente.

------ RESULTADO SPARK SQL: DESEMPEÑO POR REGIÓN ------
+------------+------------------+--------------------+--------------------------+----------------------------+
|region      |mediciones_totales|produccion_total_kwh|promedio_generacion_diaria|rendimiento_tecnico_promedio|
+------------+------------------+--------------------+--------------------------+----------------------------+
|Usulután    |10122             |296969.5            |29.34                     |4.27                        |
|Sonsonate   |9081              |296183.79           |32.62                     |4.24                        |
|La Libertad |8063              |272584.33           |33.81                     |4.25                        |
|Santa Ana   |8057              |231752.81           |28.76                     |4.27                        |
|San Miguel  |7052              |224760.21           |31.87                     |4.23                        |

In [ ]:
# 1. Borrar la carpeta previa que quedó bloqueada
!rm -rf /content/datos/analiticos/produccion_energetica.parquet

# 2. Asegurar que la sesión de Spark esté activa (si se había cerrado con spark.stop())
spark = (
    SparkSession.builder.master("local[*]")
    .appName("GreenEnergy_Cristina_Fuentes")
    .getOrCreate()
)

print("Sesión de Spark lista y carpeta limpia.")

Sesión de Spark lista y carpeta limpia.


In [ ]:
# Descargar Parquet
df_particionado = df_spark.repartition(4)
print(f"Nuevas particiones: {df_particionado.rdd.getNumPartitions()}")

ruta_parquet = DATA_ANALYTICS / "produccion_energetica.parquet"

df_particionado.write \
    .mode("overwrite") \
    .parquet(str(ruta_parquet))

print(f"Parquet guardado exitosamente en: {ruta_parquet}")

!zip -r /content/produccion_energetica_parquet.zip /content/datos/analiticos/produccion_energetica.parquet/

from google.colab import files
files.download("/content/produccion_energetica_parquet.zip")
print("¡Parquet descargado exitosamente!")

Nuevas particiones: 4
Parquet guardado exitosamente en: /content/datos/analiticos/produccion_energetica.parquet
updating: content/datos/analiticos/produccion_energetica.parquet/ (stored 0%)
updating: content/datos/analiticos/produccion_energetica.parquet/._SUCCESS.crc (stored 0%)
updating: content/datos/analiticos/produccion_energetica.parquet/_SUCCESS (stored 0%)
  adding: content/datos/analiticos/produccion_energetica.parquet/.part-00003-187b6858-27bf-409e-bba3-11dabfdee093-c000.snappy.parquet.crc (stored 0%)
  adding: content/datos/analiticos/produccion_energetica.parquet/part-00003-187b6858-27bf-409e-bba3-11dabfdee093-c000.snappy.parquet (deflated 21%)
  adding: content/datos/analiticos/produccion_energetica.parquet/part-00002-187b6858-27bf-409e-bba3-11dabfdee093-c000.snappy.parquet (deflated 20%)
  adding: content/datos/analiticos/produccion_energetica.parquet/part-00001-187b6858-27bf-409e-bba3-11dabfdee093-c000.snappy.parquet (deflated 21%)
  adding: content/datos/analiticos/prod

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

¡Parquet descargado exitosamente!


In [ ]:
# ¿Qué pregunta de negocio responde tu consulta Spark SQL, y qué resultado obtuviste
# (cita el valor o las primeras filas)?

# La consulta Spark SQL demostró que la región con mayor generación acumulada es Usulután con 296,969.50 kWh (10,122 mediciones),
# seguida muy de cerca por Sonsonate con 296,183.79 kWh (9,081 mediciones) y La Libertad con 272,584.33 kWh (8,063 mediciones).
# En contraste, San Salvador obtuvo la menor generación global (188,417.33 kWh)

In [ ]:
#  ¿Por qué Spark sería útil aquí si el dataset creciera 100 veces?

# Si el volumen de datos aumentara 100 veces (superando los 5 millones de registros), arquitecturas tradicionales en memoria simple como Pandas
# colapsarían por desbordamiento de RAM (OutOfMemory). Spark maneja ese volumen mediante procesamiento distribuido y particionado en clúster,
# evaluación perezosa (lazy evaluation) para optimizar planes de ejecución y formato columnar Parquet, evitando traer la totalidad de los datos a memoria innecesariamente.

In [ ]:
# Parte 4
import hashlib

# Matriz de Clasificacion de activos de datos
clasificacion_activos = pd.DataFrame(
    [
        [
            "produccion_kwh",
            "Métricas operativas",
            "Público",
            "Informes de sostenibilidad y generación abierta",
            "Sin restricción",
        ],
        [
            "region",
            "Ubicación macro",
            "Público",
            "Reporte geográfico agregado",
            "Sin restricción",
        ],
        [
            "temperatura_c",
            "Telemetría ambiental",
            "Interno",
            "Monitoreo operativo de paneles",
            "Acceso analistas",
        ],
        [
            "tipo_mantenimiento",
            "Operación",
            "Interno",
            "Planificación de cuadrillas",
            "Acceso analistas",
        ],
        [
            "tecnico_responsable",
            "Datos personales (PII)",
            "Confidencial",
            "Gestión interna de personal técnico",
            "Seudonimizar con Hash",
        ],
        [
            "numero_serie",
            "Identificador de hardware",
            "Confidencial",
            "Inventario y garantías de fábrica",
            "Enmascarar caracteres",
        ],
        [
            "latitud",
            "Coordenada exacta",
            "Acceso restringido",
            "Infraestructura crítica energética",
            "Redondear granularidad",
        ],
        [
            "costo_usd",
            "Dato financiero",
            "Acceso restringido",
            "Control presupuestario de mantenimiento",
            "Cifrar y restringir acceso",
        ],
    ],
    columns=["campo", "tipo", "clasificacion", "finalidad", "control"],
)

print("------ Matriz de Clasificación (4 Niveles) ------")
display(clasificacion_activos)

------ Matriz de Clasificación (4 Niveles) ------


,campo,tipo,clasificacion,finalidad,control
0,produccion_kwh,Métricas operativas,Público,Informes de sostenibilidad y generación abierta,Sin restricción
1,region,Ubicación macro,Público,Reporte geográfico agregado,Sin restricción
2,temperatura_c,Telemetría ambiental,Interno,Monitoreo operativo de paneles,Acceso analistas
3,tipo_mantenimiento,Operación,Interno,Planificación de cuadrillas,Acceso analistas
4,tecnico_responsable,Datos personales (PII),Confidencial,Gestión interna de personal técnico,Seudonimizar con Hash
5,numero_serie,Identificador de hardware,Confidencial,Inventario y garantías de fábrica,Enmascarar caracteres
6,latitud,Coordenada exacta,Acceso restringido,Infraestructura crítica energética,Redondear granularidad
7,costo_usd,Dato financiero,Acceso restringido,Control presupuestario de mantenimiento,Cifrar y restringir acceso


In [ ]:
# Técnicas de protección aplicadas en campos sensibles

# Campo 1: Seudonimización del Técnico Responsable con SHA-256 (Hashing con Salt)
SALT = "GreenEnergy-2026-Security"


def seudonimizar_tecnico(nombre):
  if pd.isna(nombre) or nombre == "Ninguno":
    return "Ninguno"
  texto = f"{SALT}|{nombre}".encode("utf-8")
  return f"TEC-{hashlib.sha256(texto).hexdigest()[:8].upper()}"


df_limpio["tecnico_token"] = df_limpio["tecnico_responsable"].apply(
    seudonimizar_tecnico
)
print("------ Seudonimización del Técnico (Hashing) ------")
print(
    df_limpio[
        df_limpio["tecnico_responsable"] != "Ninguno"
    ][["tecnico_responsable", "tecnico_token"]].drop_duplicates().head(5)
)

------ Seudonimización del Técnico (Hashing) ------
          tecnico_responsable tecnico_token
251   Rodrigo Ernesto Sánchez  TEC-AEF2FFF0
255       Diego Armando Pérez  TEC-7073C1E1
1052  Fátima Beatriz Alvarado  TEC-A9AB8CC3
1355   Claudia Patricia Rivas  TEC-D1046E01
1617       Ana Lucía Portillo  TEC-E88A297B


In [ ]:
# Campo 2: Enmascaramiento del Número de Serie (Ocultar primeros dígitos, conservar últimos 3)
def enmascarar_serie(serie):
  if pd.isna(serie) or str(serie) == "Sin registro":
    return "SN-XXXX"
  serie_str = str(serie)
  return f"SN-***{serie_str[-3:]}"


df_limpio["numero_serie_mask"] = df_limpio["numero_serie"].apply(
    enmascarar_serie
)

print("------ Verificación de Seudonimización y Enmascaramiento ------")
display(
    df_limpio[
        [
            "tecnico_responsable",
            "tecnico_token",
            "numero_serie",
            "numero_serie_mask",
        ]
    ].head(6)
)
print("\n------ Enmascaramiento de Número de Serie ------")
print(
    df_limpio[["numero_serie", "numero_serie_mask"]].drop_duplicates().head(5)
)

------ Verificación de Seudonimización y Enmascaramiento ------


,tecnico_responsable,tecnico_token,numero_serie,numero_serie_mask
0,Ninguno,Ninguno,SN311037,SN-***037
1,Ninguno,Ninguno,SN776779,SN-***779
2,Ninguno,Ninguno,SN354091,SN-***091
3,Ninguno,Ninguno,SN311037,SN-***037
4,Ninguno,Ninguno,SN590241,SN-***241
5,Ninguno,Ninguno,SN965169,SN-***169



------ Enmascaramiento de Número de Serie ------
  numero_serie numero_serie_mask
0     SN311037         SN-***037
1     SN776779         SN-***779
2     SN354091         SN-***091
4     SN590241         SN-***241
5     SN965169         SN-***169


In [ ]:
# 3. Crear vista analitica protegida
columnas_sensibles_excluidas = ["tecnico_responsable", "numero_serie"]
vista_analitica_protegida = df_limpio.drop(
    columns=columnas_sensibles_excluidas
).copy()

# Validación automática de gobernanza
expuestos = set(columnas_sensibles_excluidas).intersection(
    vista_analitica_protegida.columns
)
print("------ Validación de Identificadores Expuestos ------")
if len(expuestos) == 0:
  print(
      "ÉXITO: Ningún identificador sensible directo quedó expuesto en la vista analítica."
  )
else:
  print(f"ALERTA: Campos expuestos: {expuestos}")

------ Validación de Identificadores Expuestos ------
ÉXITO: Ningún identificador sensible directo quedó expuesto en la vista analítica.


In [ ]:
# Politica de gobernanza de datos
politica_gobernanza = """
# Política Breve de Gobernanza - GreenEnergy Analytics
1. Uso: Los datos energéticos se destinan exclusivamente al entrenamiento del modelo predictivo y optimización de la red.
2. Acceso: Se implementa principio de mínimo privilegio (RBAC); analistas operan sobre vistas protegidas con técnicos seudonimizados y números de serie enmascarados, reservando datos crudos a administradores autorizados.
3. Retención: Las mediciones de telemetría se conservarán durante 3 años en almacenamiento frío, tras lo cual los identificadores directos se purgan automáticamente de forma irreversible.
"""

print(politica_gobernanza)


# Política Breve de Gobernanza - GreenEnergy Analytics
1. Uso: Los datos energéticos se destinan exclusivamente al entrenamiento del modelo predictivo y optimización de la red.
2. Acceso: Se implementa principio de mínimo privilegio (RBAC); analistas operan sobre vistas protegidas con técnicos seudonimizados y números de serie enmascarados, reservando datos crudos a administradores autorizados.
3. Retención: Las mediciones de telemetría se conservarán durante 3 años en almacenamiento frío, tras lo cual los identificadores directos se purgan automáticamente de forma irreversible.



In [ ]:
# ¿Por qué clasificaste esos 2 campos como sensibles y qué técnica de
# enmascaramiento elegiste (ej. hashing, truncado, generalización)?

# Se clasificaron como sensibles los campos tecnico_responsable y numero_serie. El primero contiene datos personales identificables (PII) de los trabajadores,
# mientras que el segundo corresponde a identificadores de hardware que podrían facilitar sabotajes o vulnerabilidades en infraestructura crítica nacional si se filtran.

In [ ]:
#  ¿Qué riesgo mitiga esa técnica específicamente?

# Para el técnico se aplicó seudonimización determinista mediante Hashing (SHA-256 con Salt), lo que permite evaluar el impacto del personal en las reparaciones
# sin revelar su identidad real, mitigando el riesgo de exposición de privacidad y acoso laboral. Para el número de serie se utilizó enmascaramiento por sustitución parcial,
# ocultando los dígitos centrales y reteniendo únicamente los últimos tres caracteres, mitigando el riesgo de duplicación de garantías fraudulentas o trazabilidad maliciosa del inventario físico.